<a href="https://colab.research.google.com/github/TKWaititu/Job-finder/blob/main/job_finder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##**This notebook outlines python programs that scrape the popular job boards https://www.corporatestaffing.co.ke/ and https://www.myjobmag.co.ke/, and filter the jobs by looking for select keywords in the job titles and descriptions, motivated by the tedious process of going through the website page by page looking for matching jobs.**


# **CorporateStaffing jobfinder**


In [1]:
# libraries
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re

pd.set_option('display.max_rows', None)


In [2]:
def job_finder(page_url):
  # processing listings page
  page = requests.get(page_url)
  soup = BeautifulSoup(page.text)
  page_list =[]

  # finding all the jobs listed in page
  for job in soup.find_all('li', class_ = 'entry-list-item'):
    title_tag = job.find('h2',class_ = 'entry-title').a
    job_dict = {
                'job_title': title_tag.get_text(strip=True),
                'job_link': title_tag['href'],
                'date': job.find('time')['datetime']
                }
    page_list.append(job_dict)


  return page_list


In [3]:
def job_parser(job_url):
  # processing a particular job's webpage
  job_page = requests.get(job_url)
  job_soup = BeautifulSoup(job_page.text, "html.parser")

  # scraping the job description
  job_ul = job_soup.find_all('ul', class_='wp-block-list')

  job_description = []

  for ul in job_ul:
    for li in ul.find_all('li'):
        job_description.append(li.text)

  return job_description

In [4]:
def job_merger(first_page, last_page, search_terms):
  mega_list = []
  jobs_found = 0

  # processing a range of pages, finding the jobs and extracting the description
  for i in range(first_page, last_page + 1):
    page_url = f"https://www.corporatestaffing.co.ke/jobs/page/{i}"
    jobs_in_page = job_finder(page_url)
    jobs_found += len(jobs_in_page)

    for job in jobs_in_page:
      job_link = job['job_link']
      job_desc = job_parser(job_link)
      job['description'] = job_desc
      mega_list.append(job)


  # progress tracker
    print(f'Processed page {i} out of {last_page}')

  # converting the acquired data into a dataframe for cleaning
  df = pd.DataFrame(mega_list)

  # converting the list items into blocks of text
  df['description'] = df['description'].apply(
    lambda x: ' '.join(x) if isinstance(x, list) else x
)

  # removing extra whitspaces and punctuation marks
  df['description'] = (
      df['description']
      .astype(str)
      .str.strip()
      .str.replace(r'[^\w\s]', '', regex=True)
  )

  df['job_title'] = (
      df['job_title']
      .astype(str)
      .str.strip()
      .str.replace(r'[^\w\s]', '', regex=True)
  )

  # joining the title and description columns for a richer column
  df['full_desc'] = df['job_title'] + " " + df['description']

  # formatting the search terms and filtering the description column for related jobs
  pattern = '|'.join(search_terms)
  job_df = (df[df['full_desc'].str.contains(pattern, case=False, na=False)])

  print(f'Found {len(job_df)} matches out of {jobs_found} jobs.')

  return job_df

In [5]:
search_terms = ['data science', 'statistics', 'actuarial', 'data analyst', 'Finance', 'data analysis', 'financial']
job_df = job_merger(1,8, search_terms)

Processed page 1 out of 8
Processed page 2 out of 8
Processed page 3 out of 8
Processed page 4 out of 8
Processed page 5 out of 8
Processed page 6 out of 8
Processed page 7 out of 8
Processed page 8 out of 8
Found 36 matches out of 160 jobs.


In [6]:
job_df.head()

,job_title,job_link,date,description,full_desc
11,Market Access Government Affairs Manager Job ...,https://www.corporatestaffing.co.ke/job/market...,2026-03-25T08:53:45+03:00,Elevate blood and cellbased therapies as natio...,Market Access Government Affairs Manager Job ...
15,Importation Accountant Job 4560K,https://www.corporatestaffing.co.ke/job/import...,2026-03-25T08:17:44+03:00,Create and maintain SKUs in Odoo with accurate...,Importation Accountant Job 4560K Create and ma...
16,Sr Finance Compliance Investment Officer Job ...,https://www.corporatestaffing.co.ke/job/sr-fin...,2026-03-25T08:17:00+03:00,Financial Management and Reporting Financial A...,Sr Finance Compliance Investment Officer Job ...
17,Senior Credit Officer Job Finnlemm Sacco,https://www.corporatestaffing.co.ke/job/senior...,2026-03-25T08:16:16+03:00,Credit Monitoring Recovery Credit Reporting a...,Senior Credit Officer Job Finnlemm Sacco Credi...
18,General Manager Logistics Tanzania Job,https://www.corporatestaffing.co.ke/job/genera...,2026-03-25T08:15:26+03:00,Develop and execute the companys strategic pla...,General Manager Logistics Tanzania Job Develop...


In [7]:
for title, link, date in zip(job_df['job_title'], job_df['job_link'], job_df['date']):
    print(f"Title: {title}\nLink: {link}\nDate: {date}\n")

Title: Market Access  Government Affairs Manager Job Terumo BCT
Link: https://www.corporatestaffing.co.ke/job/market-access-government-affairs-manager-job-terumo-bct/
Date: 2026-03-25T08:53:45+03:00

Title: Importation Accountant Job 4560K
Link: https://www.corporatestaffing.co.ke/job/importation-accountant-job-45-60k/
Date: 2026-03-25T08:17:44+03:00

Title: Sr Finance Compliance  Investment Officer Job Finnlemm Sacco
Link: https://www.corporatestaffing.co.ke/job/sr-finance-compliance-investment-officer-job-finnlemm-sacco/
Date: 2026-03-25T08:17:00+03:00

Title: Senior Credit Officer Job Finnlemm Sacco
Link: https://www.corporatestaffing.co.ke/job/senior-credit-officer-job-finnlemm-sacco/
Date: 2026-03-25T08:16:16+03:00

Title: General Manager Logistics Tanzania Job
Link: https://www.corporatestaffing.co.ke/job/general-manager-logistics-tanzania-job/
Date: 2026-03-25T08:15:26+03:00

Title: Business Intelligence Manager Job MKOPA Solar
Link: https://www.corporatestaffing.co.ke/job/busin

# **MyjobMag jobfinder**


In [8]:
def job_finder_mjm(page_url):
  # processing listings page
  page = requests.get(page_url)
  soup = BeautifulSoup(page.text)
  page_list =[]
  # finding all the jobs listed in page
  for job in soup.find_all('li', class_ = 'job-list-li'):
    try:
      h_tag = job.find('h2').a
      time_tag = job.find('li', class_ = 'job-item').ul

      job_dict = {
                  'job_title': h_tag.get_text(strip=True),
                  'job_link': 'https://www.myjobmag.co.ke' + h_tag['href'],
                  'date': time_tag.get_text(strip =True)
                  }
      page_list.append(job_dict)
    except:
      continue


  return page_list


In [9]:
d = job_finder_mjm('https://www.myjobmag.co.ke/page/1')

In [10]:
def job_parser_mjm(job_url):
  # processing a particular job's webpage
  job_page = requests.get(job_url)
  job_soup = BeautifulSoup(job_page.text, "html.parser")

  # scraping the job description
  job_ul = job_soup.find('div', class_='job-details')

  job_description = []

  for ul in job_ul:
    for li in ul.find_all('li'):
        job_description.append(li.text)

  return job_description

In [11]:
def job_parser_mjm(job_url):
  # processing a particular job's webpage
  job_page = requests.get(job_url)
  job_soup = BeautifulSoup(job_page.text, "html.parser")

  # scraping the job description
  job_ul = job_soup.find('div', class_='job-details')

  job_description = [job_ul.get_text(strip = True)]

  return job_description

In [12]:
def job_merger_mjm(first_page, last_page, search_terms):
  mega_list = []
  jobs_found = 0

  # processing a range of pages, finding the jobs and extracting the description
  for i in range(first_page, last_page + 1):
    page_url = f"https://www.myjobmag.co.ke/page/{i}"
    jobs_in_page = job_finder_mjm(page_url)
    jobs_found += len(jobs_in_page)

    for job in jobs_in_page:
      job_link = job['job_link']
      job_desc = job_parser_mjm(job_link)
      job['description'] = job_desc
      mega_list.append(job)


  # progress tracker
    print(f'Processed page {i} out of {last_page}')

  # converting the acquired data into a dataframe for cleaning
  df = pd.DataFrame(mega_list)

  # converting the list items into blocks of text
  df['description'] = df['description'].apply(
    lambda x: ' '.join(x) if isinstance(x, list) else x
)

  # removing extra whitspaces and punctuation marks
  df['description'] = (
      df['description']
      .astype(str)
      .str.strip()
      .str.replace(r'[^\w\s]', '', regex=True)
  )

  df['job_title'] = (
      df['job_title']
      .astype(str)
      .str.strip()
      .str.replace(r'[^\w\s]', '', regex=True)
  )

  # joining the title and description columns for a richer column
  df['full_desc'] = df['job_title'] + " " + df['description']

  # formatting the search terms and filtering the description column for related jobs
  pattern = '|'.join(search_terms)
  job_df = (df[df['full_desc'].str.contains(pattern, case=False, na=False)])

  print(f'Found {len(job_df)} matches out of {jobs_found} jobs.')

  return job_df

In [13]:
search_terms = ['data science', 'statistics', 'actuarial', 'data analyst', 'Finance', 'data analysis', 'financial']
job_df = job_merger_mjm(3,7, search_terms)

Processed page 3 out of 7
Processed page 4 out of 7
Processed page 5 out of 7
Processed page 6 out of 7
Processed page 7 out of 7
Found 45 matches out of 125 jobs.


In [14]:
for title, link, date in zip(job_df['job_title'], job_df['job_link'], job_df['date']):
    print(f"Title: {title}\nLink: {link}\nDate: {date}\n")

Title: Anthropology Unit Lead at Medecins Sans Frontieres MSF
Link: https://www.myjobmag.co.ke/job/anthropology-unit-lead-medecins-sans-frontieres-msf
Date: 24 March

Title: Vacancies at International Rescue Committee
Link: https://www.myjobmag.co.ke/jobs/vacancies-at-international-rescue-committee-47
Date: 24 March

Title: Finance Manager at Childcare Worldwide
Link: https://www.myjobmag.co.ke/job/finance-manager-childcare-worldwide
Date: 24 March

Title: Treasury  Investment Manager at Inkomoko
Link: https://www.myjobmag.co.ke/job/treasury-investment-manager-inkomoko
Date: 24 March

Title: Jobs at Eleven Degrees Consulting
Link: https://www.myjobmag.co.ke/jobs/jobs-at-eleven-degrees-consulting
Date: 24 March

Title: Chief Officers at County Government of Nyeri
Link: https://www.myjobmag.co.ke/jobs/chief-officers-at-county-government-of-nyeri
Date: 24 March

Title: Administrator III Data Operations at TransUnion
Link: https://www.myjobmag.co.ke/job/administrator-iii-data-operations-tr